In [4]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd

# ----------------------------------------------------
# 1. Read input CSVs
#    Change these names to match your actual files
# ----------------------------------------------------
index_a_path = "data/index_A.csv"      # e.g., "index_A (3).csv"
heat_path    = "data/Heat.csv"         # Heat dataframe file
A_path       = "data/A.csv"            # e.g., "A (1).csv"

Index_A = pd.read_csv(index_a_path)   # must contain column: "Processes"
Heat    = pd.read_csv(heat_path)      # must contain column: "Processes"
A       = pd.read_csv(A_path)         # numeric matrix aligned with Index_A rows

# If A contains a non-numeric column like "Processes", drop it before analysis:
# (Uncomment if needed)
# A_numeric = A.select_dtypes(include="number")
# A = A_numeric

# Ensure Index_A has a clean integer index (0,1,2,...)
Index_A = Index_A.reset_index(drop=True)

# ----------------------------------------------------
# 2. Map process name → row index in Index_A
# ----------------------------------------------------
if "Processes" not in Index_A.columns:
    raise KeyError("Index_A must have a 'Processes' column")

if "Processes" not in Heat.columns:
    raise KeyError("Heat must have a 'Processes' column")

# Create mapping from process name to its row index in Index_A
proc_to_idx = (
    Index_A.reset_index()              # temporary column 'index'
           .set_index("Processes")["index"]
)

# For each process in Heat, find its corresponding index in Index_A
matched_indices = (
    Heat["Processes"]
    .map(proc_to_idx)                  # may give NaN if not found
    .dropna()
    .astype(int)
    .unique()
)

# If none matched, bail out early
if len(matched_indices) == 0:
    print("No processes from Heat matched Index_A['Processes'].")
    result_df = pd.DataFrame(columns=["Processes"])
    result_df.to_csv("matched_processes.csv", index=False)
    raise SystemExit

# ----------------------------------------------------
# 3. In A, for those row indices, find non-zero columns
# ----------------------------------------------------
# Assumption:
#   - Rows of A are aligned with rows of Index_A
#   - Columns of A either:
#       (a) are integer indices that refer to Index_A rows, OR
#       (b) are process names that we can match to Index_A['Processes'].
#
# Code below handles both options.

nonzero_target_indices = set()   # these will be indices in Index_A

for r_idx in matched_indices:
    # Get the entire row from A
    row = A.iloc[r_idx]

    # Find columns where values are non-zero (ignoring NaN)
    nonzero_cols = row[(row != 0) & row.notna()].index

    for col_name in nonzero_cols:
        # Case 1: column name is an integer index into Index_A
        try:
            idx = int(col_name)
            nonzero_target_indices.add(idx)
            continue
        except (ValueError, TypeError):
            pass

        # Case 2: column name is a process name that appears in Index_A["Processes"]
        if col_name in proc_to_idx.index:
            idx = int(proc_to_idx[col_name])
            nonzero_target_indices.add(idx)
        else:
            # If neither, we just skip this column
            # (you can print a warning if you want)
            # print(f"Warning: column '{col_name}' not recognized as index or process name")
            pass

# Deduplicate and keep only valid indices in Index_A range
nonzero_target_indices = sorted(
    i for i in nonzero_target_indices
    if 0 <= i < len(Index_A)
)

# ----------------------------------------------------
# 4. Come back to Index_A, get process names for those indices
# ----------------------------------------------------
if len(nonzero_target_indices) == 0:
    print("No non-zero entries found in A for the matched rows.")
    result_df = pd.DataFrame(columns=["Processes"])
else:
    result_df = (
        Index_A.loc[nonzero_target_indices, ["Processes"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

# Print result
print("Matched processes with non-zero connections in A:")
print(result_df)

# ----------------------------------------------------
# 5. Save to CSV
# ----------------------------------------------------
output_path = "matched_processes.csv"
result_df.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")
